# Text Vectorization Using TF-IDF and Similarity Analysis

**Input:** `../data/processed/ted_talks_cleaned.csv` (4,005 talks, 16 columns)
**Outputs:**
- `../data/processed/tfidf_matrix.npz`
- `../data/processed/cosine_sim.npy`
- `../data/processed/tfidf_vectorizer.joblib`

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

pd.set_option("display.max_colwidth", 120)
sns.set_style("whitegrid")

In [3]:
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

Shape: (4005, 16)
Columns: ['talk_id', 'title', 'speaker', 'speaker_occupation', 'description', 'tags', 'transcript', 'url', 'views', 'duration', 'published_date', 'event', 'transcript_word_count', 'is_short_transcript', 'combined_text', 'combined_text_prefixed']


,talk_id,title,speaker,speaker_occupation,description,tags,transcript,url,views,duration,published_date,event,transcript_word_count,is_short_transcript,combined_text,combined_text_prefixed
0,1,averting the climate crisis,Al Gore,climate advocate,with the same humor and humanity he exuded in an inconvenient truth al gore spells out 15 ways that individuals can ...,"alternative energy, cars, climate change, culture, environment, global issues, science, sustainability, technology",thank you so much chris and it s truly a great honor to have the opportunity to come to this stage twice i m extreme...,https://www.ted.com/talks/al_gore_averting_the_climate_crisis/,3523392,977,2006-06-27,TED2006,2186,False,averting the climate crisis alternative energy cars climate change culture environment global issues science sustain...,title: averting the climate crisis tags: alternative energy cars climate change culture environment global issues sc...
1,92,the best stats you ve ever seen,Hans Rosling,global health expert; data visionary,you ve never seen data presented like this with the drama and urgency of a sportscaster statistics guru hans rosling...,"africa, asia, google, demo, economics, global issues, health, statistics, global development, visualizations, math",about 10 years ago i took on the task to teach global development to swedish undergraduate students that was after h...,https://www.ted.com/talks/hans_rosling_the_best_stats_you_ve_ever_seen/,14501685,1190,2006-06-27,TED2006,3216,False,the best stats you ve ever seen africa asia google demo economics global issues health statistics global development...,title: the best stats you ve ever seen tags: africa asia google demo economics global issues health statistics globa...
2,7,simplicity sells,David Pogue,technology columnist,new york times columnist david pogue takes aim at technology s worst interface design offenders and provides encoura...,"computers, entertainment, interface design, media, music, performance, simplicity, software, technology",hello voice mail my old friend i ve called for tech support again i ignored my boss s warning i called on a monday m...,https://www.ted.com/talks/david_pogue_simplicity_sells/,1920832,1286,2006-06-27,TED2006,3444,False,simplicity sells computers entertainment interface design media music performance simplicity software technology new...,title: simplicity sells tags: computers entertainment interface design media music performance simplicity software t...


##### Review the cleaned dataset

In [4]:
print("Rows:", len(df))
print("Non-empty combined_text:", (df["combined_text"].fillna("") != "").sum())
print("Non-empty combined_text_prefixed:", (df["combined_text_prefixed"].fillna("") != "").sum())
print("Short-transcript talks flagged:", df["is_short_transcript"].sum())

# Drop any rows with empty prefixed text (safety)
df = df[df["combined_text_prefixed"].fillna("") != ""].reset_index(drop=True)
print("Usable rows after filter:", len(df))

Rows: 4005
Non-empty combined_text: 4005
Non-empty combined_text_prefixed: 4005
Short-transcript talks flagged: 10
Usable rows after filter: 4005


##### TF-IDF vectorization 

`TfidfVectorizer` with:
- `stop_words="english"` - drop very common words (`the`, `and`, …).
- `ngram_range=(1, 2)` - capture both single words and short phrases (e.g. "climate change").
- `min_df=2` - ignore terms that appear in only 1 talk (reduces noise, vocabulary size).
- `max_df=0.9` - ignore terms that appear in > 90 % of talks (near-stopwords).
- `TF-IDF(t, d) = TF(t, d) × IDF(t)`

##### Fit TF-IDF

In [5]:
t0 = time.time()

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
)

tfidf_matrix = vectorizer.fit_transform(df["combined_text_prefixed"])
vocab = vectorizer.get_feature_names_out()

print(f"TF-IDF built in {time.time()-t0:.1f}s")
print("Matrix shape (docs × terms):", tfidf_matrix.shape)
print("Vocabulary size:", len(vocab))
print("Non-zero entries:", tfidf_matrix.nnz)

TF-IDF built in 11.7s
Matrix shape (docs × terms): (4005, 366288)
Vocabulary size: 366288
Non-zero entries: 3123292


In [6]:
# Vocabulary sample
print("First 30 vocab terms:", list(vocab[:30]))
print("\nRandom 30 vocab terms:", list(np.random.choice(vocab, 30, replace=False)))

First 30 vocab terms: ['00', '00 morning', '000', '000 000', '000 10', '000 100', '000 11', '000 12', '000 15', '000 18', '000 20', '000 200', '000 300', '000 40', '000 50', '000 500', '000 60', '000 70', '000 750', '000 acre', '000 acres', '000 active', '000 actually', '000 ads', '000 adults', '000 age', '000 american', '000 americans', '000 animals', '000 ants']

Random 30 vocab terms: ['washington called', 'array sensors', 'mortality death', 'children thousands', '1959', 'diamond shape', 'kind literally', 'census', 'electrons flowing', 'right graph', 'northwest amazon', 'delivery hospital', 'quote able', 'walk started', 'profound live', 'having special', 'important drive', 'reinforce power', 'happiest man', 'like day', 'arrangements', 'capability thing', 'bands', 'word working', 'just massive', 'exchanging information', 'light sugar', 'end big', 'said patient', 'presumably']


In [7]:
# Top terms for one talk
def top_terms_for_doc(idx, k=10):
    row = tfidf_matrix[idx].toarray().ravel()
    top_idx = row.argsort()[::-1][:k]
    return [(vocab[i], round(float(row[i]), 4)) for i in top_idx if row[i] > 0]

sample_idx = 0
print("Talk:", df.loc[sample_idx, "title"])
print("Top 10 TF-IDF terms:")
for term, w in top_terms_for_doc(sample_idx, 10):
    print(f"  {term:35s}  {w}")

Talk: averting the climate crisis
Top 10 TF-IDF terms:
  tipper                               0.2052
  carbon                               0.1426
  cars trucks                          0.1178
  nashville                            0.1178
  slideshow                            0.1151
  climate                              0.1093
  cost family                          0.1063
  family restaurant                    0.1063
  climate crisis                       0.1032
  global warming                       0.0982


##### Cosine similarity

Cosine similarity compares two vectors by the **angle** between them:
- `1.0` = identical direction (same term distribution).
- `0.0` = orthogonal (no shared weighted terms).

Each talk is compared against every other talk → a `N × N` matrix. We **set the diagonal to 0** because a talk should never recommend itself (`cosine(talk_i, talk_i) = 1.0` always).

In [8]:
# Compute + save similarity matrix

t0 = time.time()
cos_sim = cosine_similarity(tfidf_matrix, dense_output=True)
np.fill_diagonal(cos_sim, 0.0)
print(f"Similarity matrix computed in {time.time()-t0:.1f}s")
print("Shape:", cos_sim.shape)
print("Value range:", float(cos_sim.min()), "→", float(cos_sim.max()))

Similarity matrix computed in 2.7s
Shape: (4005, 4005)
Value range: 0.0 → 0.9784187317800085


In [9]:
# Example similarity lookups
i = 0  # first talk (Al Gore, climate crisis)
top_idx = cos_sim[i].argsort()[::-1][:5]

print("Seed talk:", df.loc[i, "title"])
print("Top 5 most similar (self excluded):")
for j in top_idx:
    print(f"  {cos_sim[i, j]:.4f}  {df.loc[j, 'title']}")

Seed talk: averting the climate crisis
Top 5 most similar (self excluded):
  0.1748  new thinking on the climate crisis
  0.1643  how we can turn the tide on climate
  0.1433  a climate solution where all sides can win
  0.1423  the case for optimism on climate change
  0.1382  can we stop climate change by removing co2 from the air


##### Sample recommendations

Run `recommend(talk_id, k=5)` for **3 different seed talks** (climate, education/creativity, tech/design). For each, record titles and scores and manually judge relevance.

In [12]:
# Recommendation helper
def recommend(talk_id, k=5):
    """Return top-k most similar talks for a given talk_id (self excluded)."""
    matches = df.index[df["talk_id"] == talk_id]
    if len(matches) == 0:
        return None
    i = int(matches[0])
    scores = cos_sim[i]
    top_idx = scores.argsort()[::-1][:k]
    rows = []
    for rank, j in enumerate(top_idx, 1):
        rows.append({
            "rank": rank,
            "talk_id": int(df.loc[j, "talk_id"]),
            "title": df.loc[j, "title"],
            "score": round(float(scores[j]), 4),
        })
    out = pd.DataFrame(rows)
    out.insert(0, "seed_id", talk_id)
    out.insert(1, "seed_title", df.loc[i, "title"])
    return out

In [13]:
# Three seed recommendations
for seed_id in [1, 66, 7]:
    print(f"\n========== Recommendations for talk_id {seed_id} ==========")
    rec = recommend(seed_id, k=5)
    print(rec.to_string(index=False))


========== Recommendations for talk_id 1 ==========
 seed_id                  seed_title  rank  talk_id                                                   title  score
       1 averting the climate crisis     1      243                      new thinking on the climate crisis 0.1748
       1 averting the climate crisis     2    54715                     how we can turn the tide on climate 0.1643
       1 averting the climate crisis     3     2784              a climate solution where all sides can win 0.1433
       1 averting the climate crisis     4     2441                 the case for optimism on climate change 0.1423
       1 averting the climate crisis     5     3579 can we stop climate change by removing co2 from the air 0.1382

========== Recommendations for talk_id 66 ==========
 seed_id                 seed_title  rank  talk_id                                              title  score
      66 do schools kill creativity     1     1738             how to escape education s death

In [15]:
# Quick manual relevance notes — record your observations here
notes = {
    1:  "All 5 recs are climate talks. Strong cluster. Score gap between seed and top match is small (0.17) — TF-IDF is conservative on long transcripts.",
    66: "Ranks 1-2 are education/creativity (Ken Robinson's own follow-ups). Ranks 3-5 drift into 'rebuilding', 'overwhelmed', 'company' — generic words cause drift.",
    7:  "Design/tech cluster holds. 'designing for simplicity' and 'science of simplicity' are near-perfect matches. 'save the internet' and 'electronic spreadsheet' are tangentially relevant (tech theme).",
}
for k, v in notes.items():
    print(f"talk_id {k}: {v}\n")

talk_id 1: All 5 recs are climate talks. Strong cluster. Score gap between seed and top match is small (0.17) — TF-IDF is conservative on long transcripts.

talk_id 66: Ranks 1-2 are education/creativity (Ken Robinson's own follow-ups). Ranks 3-5 drift into 'rebuilding', 'overwhelmed', 'company' — generic words cause drift.

talk_id 7: Design/tech cluster holds. 'designing for simplicity' and 'science of simplicity' are near-perfect matches. 'save the internet' and 'electronic spreadsheet' are tangentially relevant (tech theme).



##### Limitations observed

1. **Frequent names beat topics.** The top-weighted term for the climate talk was `tipper` (Al Gore's wife), not `climate`. TF-IDF rewards words that repeat a lot *and* are rare elsewhere - so a name said fifty times can outrank the actual topic. 
2. **Numbers leak in.** Vocabulary contains `000`, `000 morning`, `000 100` … because Task 3 didn't strip digits. They carry no meaning and inflate the 366k-term matrix. Stripping numbers upstream would reduce noise and matrix size.
3. **Off-topic drift on seed 66.** Ranks 3–5 for *Do schools kill creativity?* were about rebuilding a neighborhood, feeling overwhelmed, and running a company. They share generic phrases ("rethink", "people") but different topics - TF-IDF can't tell that "rethink education" and "rethink a company" mean different things.
4. **Scores are low but relative order is what matters.** 0.13–0.17 looks weak on an absolute scale, but that's normal when transcripts are long and the vocab is huge. A fixed threshold ("≥ 0.5 = good") would return nothing. Judge by ranking, not by score value.
5. **Popularity is ignored.** Cosine has no idea Al Gore has 3.5M views and an obscure climate talk has 40k. "Best match" ≠ "best talk" from a user's perspective.
6. **Short transcripts score lower by design.** The 10 music/performance talks only have title + tags to match on, so their similarity is structurally smaller even when correct.

## TF-IDF and cosine similarity

**TF-IDF:** A word's weight goes up when it appears many times in one talk and down when it appears in many talks. So "carbon" scores high for a climate talk (repeated there, rare elsewhere) while "people" scores low (everywhere). Each talk becomes a sparse vector where only its distinctive words matter.

**Cosine similarity:** Compares the *angle* between two of those vectors, ignoring length. Two talks using the same distinctive vocabulary point the same way → score near 1. Unrelated talks point elsewhere → score near 0.

**Ranking:** For a seed talk, score it against every other talk, sort descending, drop itself, take top-k. That's the recommendation list.

In [16]:
# Save the TF-IDF matrix, similarity matrix, and vectorizer for later use

# os.makedirs("../data/processed", exist_ok=True)
from scipy.sparse import save_npz

save_npz("../data/processed/tfidf_matrix.npz", tfidf_matrix)
np.save("../data/processed/cosine_sim.npy", cos_sim)
joblib.dump(vectorizer, "../data/processed/tfidf_vectorizer.joblib")

print("Saved:")
print("  tfidf_matrix.npz         ", tfidf_matrix.shape)
print("  cosine_sim.npy           ", cos_sim.shape, cos_sim.dtype)
print("  tfidf_vectorizer.joblib  ")

Saved:
  tfidf_matrix.npz          (4005, 366288)
  cosine_sim.npy            (4005, 4005) float64
  tfidf_vectorizer.joblib  


## Short Findings Report

**Artefacts**
- `tfidf_matrix.npz` - sparse 4005 × 366,288 TF-IDF matrix
- `cosine_sim.npy` - dense 4005 × 4005 cosine matrix (self-similarity = 0)
- `tfidf_vectorizer.joblib` - reusable vectorizer for Task 5 / Streamlit

**Config:** `TfidfVectorizer(stop_words="english", ngram_range=(1,2), min_df=2, max_df=0.9)` on `combined_text_prefixed`.

**What worked**
- Climate seed -> 5/5 climate talks.
- Creativity seed -> 2 strong education hits (Ken Robinson's own sequels).
- Simplicity seed -> design/tech cluster holds; two near-perfect "simplicity" matches.

**What broke**
- Education seed drifted off-topic by rank 3 (generic words like "rethink").
- A person's name (`tipper`) outweighted the actual topic.
- Number tokens polluted vocab.

**Next (Task 5)**
Wrap `recommend()` into a module, tune field weighting (tags ×2–3 or drop transcript weight), evaluate on 10+ seeds, then plug into Streamlit.